# Pump.fun Trading Strategy — Full Analysis Pipeline

**Complete circle:** download replay data → run trading engine → export → XGBoost/SHAP/statistical analysis → Optuna weight optimization → report.

Run cells in order. No GPU required — runs on free Colab CPU.

In [ ]:
# Cell 1: Install dependencies (~2 min)
!pip install -q zstandard pandas numpy scipy scikit-learn xgboost shap optuna matplotlib plotly statsmodels
print("Dependencies installed.")

In [ ]:
# Cell 2: Upload the analysis scripts or clone repo
import os, sys, urllib.request, zipfile, io

# Option A: If you uploaded scripts/analysis/ to Colab, skip this cell
# Option B: Clone from GitHub (replace with your repo URL)
# !git clone https://github.com/YOUR_USER/YOUR_REPO.git
# %cd YOUR_REPO/scripts/analysis

# Option C: Upload from local machine (recommended for first run)
from google.colab import files
print("Upload the analysis folder as a zip, or upload individual files.")
print("See the README for folder structure.")

# Check if we have the modules
try:
    import config
    from runner import ReplayRunner
    from analysis import run_analysis, print_report, optuna_optimize, load_trades_from_csv
    from pipeline import export_to_csv
    print("Modules loaded successfully.")
except ImportError:
    print("Modules not found. Upload the analysis folder or provide a path.")

In [ ]:
# Cell 3: Build and run the trading engine on replay data
import time

HOURS = 2  # change for more/fewer replay hours

runner = ReplayRunner(sol_balance=10.0, sol_amount=0.01)
runner.download_and_replay_recent(HOURS)
runner.print_summary()

result = runner.get_result()
trades = result['trades']
rejected = result['rejected']
print(f"\nTrades: {len(trades)}  Rejected signals: {len(rejected)}")

In [ ]:
# Cell 4: Export trades and rejected signals to CSV
paths = export_to_csv(trades, rejected)
print(f"Trades CSV: {paths.get('trades', 'N/A')}")
print(f"Rejected CSV: {paths.get('rejected', 'N/A')}")

# Show summary table
import pandas as pd
if 'trades' in paths:
    df = pd.read_csv(paths['trades'])
    print(f"\n{len(df)} trades columns:\n{list(df.columns)}")
    display(df.head())

In [ ]:
# Cell 5: Full analysis — XGBoost + SHAP + feature importance
df = load_trades_from_csv(paths['trades'])

from analysis import avail_features
if len(avail_features(df)) >= 2:
    result = run_analysis(paths['trades'], paths.get('rejected'))
    print_report(result)
else:
    print("Not enough trades with feature data for ML analysis.")

In [ ]:
# Cell 6: SHAP per-trade explanation (shows WHY each trade scored what it did)
import shap
import matplotlib.pyplot as plt
import numpy as np

if 'result' in locals() and result.get('xgb') and result.get('shap'):
    model = result['xgb']['model']
    X_test = result['xgb']['X_test']
    feature_names = result['xgb']['feature_names']

    # Global SHAP summary
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)

    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_test, feature_names=feature_names, show=False)
    plt.title('SHAP Feature Impact on Win Prediction')
    plt.tight_layout()
    plt.show()

    # SHAP bar
    plt.figure(figsize=(8, 4))
    shap.bar_plot(np.abs(shap_values).mean(axis=0), feature_names=feature_names)
    plt.title('Mean |SHAP| — Global Feature Importance')
    plt.tight_layout()
    plt.show()

    # Waterfall for first test trade
    shap.plots.waterfall(shap.Explanation(
        values=shap_values[0],
        base_values=explainer.expected_value,
        data=X_test[0],
        feature_names=feature_names
    ), show=False)
    plt.title('SHAP Waterfall — Trade #0 Explanation')
    plt.tight_layout()
    plt.show()
else:
    print("Run Cell 5 first to generate XGBoost results.")

In [ ]:
# Cell 7: Optuna automated weight optimization
N_TRIALS = 500  # increase for better results (trade-off: time)

opt_result = optuna_optimize(df, n_trials=N_TRIALS)
if opt_result:
    print(f"Best PF achieved: {opt_result['best_value']:.2f}")
    print(f"\nCurrent v4 weights:")
    print(f"  wallet = 0.40,  signalAge = 0.45,  liquidity = 0.15")
    print(f"\nOptuna optimal weights (normalized):")
    b = opt_result['best_params_norm']
    print(f"  wallet = {b['wallet']:.3f},  signalAge = {b['age']:.3f},  liquidity = {b['liq']:.3f}")
    print(f"\nImprovement potential: +{(opt_result['best_value'] - result.get('pf', 0)):.2f} PF")
else:
    print("Not enough data for Optuna optimization (need 10+ trades).")

In [ ]:
# Cell 8: Bucket analysis — wallet count, signal age, buy ratio
from analysis import bucket_analysis

buckets = bucket_analysis(df)

for key, title in [('score_buckets', 'Score Buckets'),
                    ('wallet_buckets', 'Wallet Buckets'),
                    ('age_buckets', 'Signal Age Buckets'),
                    ('exit_reasons', 'Exit Reasons')]:
    bdf = buckets.get(key)
    if bdf is not None and len(bdf):
        print(f"\n{title}:")
        display(bdf)

In [ ]:
# Cell 9: Plot equity curve
import matplotlib.pyplot as plt

if len(trades) > 1:
    sorted_trades = sorted(trades, key=lambda t: t.exit_time)
    cum_pnl = []
    running = 0
    for t in sorted_trades:
        running += t.pnl
        cum_pnl.append(running)

    plt.figure(figsize=(12, 4))
    plt.plot(cum_pnl, marker='.', linestyle='-', linewidth=1)
    plt.axhline(0, color='gray', linestyle='--', alpha=0.5)
    plt.title('Equity Curve')
    plt.xlabel('Trade #')
    plt.ylabel('Cumulative PnL ($)')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Not enough trades for equity curve.")

In [ ]:
# Cell 10: Download CSVs for local analysis
from google.colab import files
import os

csv_dir = '/content/'
if 'paths' in locals():
    for p in paths.values():
        if os.path.exists(p):
            files.download(p)
            print(f"Downloaded: {p}")

print("\nDone! All analysis files are available in the Colab file browser.")